# Coordinated Verbs and Auxiliaries

In [1]:
#first parse the corpus
# Import the toolkit, giving it a shorter alias 'mptf'
import mptf_parser as mptf

# Set folder paths
input_folder = r"C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026"
output_folder = r"C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output"

In [2]:
import os
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS
import mptf_parser as mptf

# ======================================================
# LOAD BOTH CORPORA
# ======================================================

print("\n--- Loading corpora ---")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers["id"] = lambda line, i: line[i]
custom_field_parsers["head"] = lambda line, i: line[i]

my_corpus = []
syntactically_annotated_corpus = []

conllu_files = sorted(
    f for f in os.listdir(output_folder)
    if f.lower().endswith(".conllu")
    and "outdated" not in f.lower()
)

for filename in conllu_files:
    file_path = os.path.join(output_folder, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = f.read()

    # --- Clean malformed lines ---
    lines = raw_data.splitlines()
    clean_lines = [
        line for line in lines
        if line.startswith("#")
        or line.strip() == ""
        or line.count("\t") == 9
    ]
    clean_data = "\n".join(clean_lines) + "\n"

    # --- Parse sentences ---
    sentences = parse(clean_data, field_parsers=custom_field_parsers)

    for sent in sentences:
        sentence_obj = mptf.Sentence(
            sent,
            source_filename=filename
        )
        sentence_obj.metadata["source_filename"] = filename

        # → always add to the full corpus
        my_corpus.append(sentence_obj)

        # → add to syntactic corpus ONLY if deprel exists
        if any(tok.get("deprel") not in (None, "_") for tok in sent):
            syntactically_annotated_corpus.append(sentence_obj)

# ======================================================
# CONFIRMATION
# ======================================================

print("✔ Corpora loaded successfully.")
print(f"  my_corpus (all texts):")
print(f"    texts:     {len(set(s.metadata['source_filename'] for s in my_corpus))}")
print(f"    sentences: {len(my_corpus)}")

print(f"\n  syntactically_annotated_corpus:")
print(f"    texts:     {len(set(s.metadata['source_filename'] for s in syntactically_annotated_corpus))}")
print(f"    sentences: {len(syntactically_annotated_corpus)}")


--- Loading corpora ---
✔ Corpora loaded successfully.
  my_corpus (all texts):
    texts:     42
    sentences: 33471

  syntactically_annotated_corpus:
    texts:     31
    sentences: 5451


In [3]:
from collections import defaultdict
import pandas as pd

records = []

for sentence in syntactically_annotated_corpus:
    file_id = sentence.metadata.get("source_filename", "unknown")
    sent_id = sentence.metadata.get("sent_id", "unknown")
    
    tokens = sentence.get_tokens()
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:
        # Check conj participle
        if token.deprel == "conj" and token.upos == "VERB":
            if not token.feats:
                continue
            if token.feats.get("VerbForm") != "Part":
                continue

            # Get head
            head_id = str(token.head)
            head = token_by_id.get(head_id)
            if not head:
                continue

            # Head must also be participle verb
            if head.upos == "VERB" and head.feats and head.feats.get("VerbForm") == "Part":
                
                records.append({
                    "File": file_id,
                    "SentenceID": sent_id,
                    "Participle_conj": token.lemma,
                    "Participle_head": head.lemma,
                    "Deprel": token.deprel
                })

# Create DataFrame
df_conj_parts = pd.DataFrame(records)

print("\n=== Participle conj → participle head ===")
print(df_conj_parts.head(20))

print("\nTotal cases:", len(df_conj_parts))


=== Participle conj → participle head ===
                   File SentenceID Participle_conj Participle_head Deprel
0    DD-K35_mptf.conllu         44          guftan           dādan   conj
1    DD-K35_mptf.conllu        224       payrāstan          kirdan   conj
2     Dk5-B_mptf.conllu         12       payrāstan         tāšīdan   conj
3     Dk5-B_mptf.conllu        358      padīriftan         wizīdan   conj
4     Dk5-B_mptf.conllu        397        gumēxtan        gumēxtan   conj
5     Dk7-B_mptf.conllu         23    fragānēnīdan          kirdan   conj
6   GBd-TD1_mptf.conllu        252         raftan1         ēstādan   conj
7   GBd-TD1_mptf.conllu        287       paywastan       paywastan   conj
8   GBd-TD1_mptf.conllu        287       paywastan       paywastan   conj
9   GBd-TD1_mptf.conllu        387          madan1          madan1   conj
10  GBd-TD1_mptf.conllu        387          madan1          madan1   conj
11  GBd-TD1_mptf.conllu        387          madan1          madan1   

In [4]:
import pandas as pd
from collections import defaultdict, Counter
import os

# ------------------------------
# Storage
# ------------------------------
records = []
deprel_counter = Counter()

# ------------------------------
# Helper: climb conj chain
# ------------------------------
def get_non_conj_head(token, token_by_id):
    current = token
    while True:
        head_id = str(current.head)
        if head_id not in token_by_id:
            return None
        head = token_by_id[head_id]

        # stop if head is not conj
        if head.deprel != "conj":
            return head
        else:
            current = head  # climb higher

# ------------------------------
# Process syntactic corpus
# ------------------------------
for sentence in syntactically_annotated_corpus:
    file_id = sentence.metadata.get("source_filename", "unknown")
    tokens = sentence.get_tokens()
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:
        # Condition 1: token is conj participle
        if token.deprel != "conj":
            continue
        if token.upos != "VERB":
            continue
        if not token.feats or token.feats.get("VerbForm") != "Part":
            continue

        # Get head
        head_id = str(token.head)
        if head_id not in token_by_id:
            continue
        head = token_by_id[head_id]

        # Condition 2: head also participle verb
        if head.upos != "VERB":
            continue
        if not head.feats or head.feats.get("VerbForm") != "Part":
            continue

        # Climb conj chain
        top_head = get_non_conj_head(head, token_by_id)
        if not top_head:
            continue

        head_deprel = top_head.deprel
        deprel_counter[head_deprel] += 1

        # Record
        records.append({
            "File": file_id,
            "TokenLemma": token.lemma,
            "HeadLemma": head.lemma,
            "TopHeadLemma": top_head.lemma,
            "TopHeadDeprel": head_deprel
        })

# ------------------------------
# DataFrame
# ------------------------------
df = pd.DataFrame(records)

# ------------------------------
# Deprel statistics
# ------------------------------
df_deprel_stats = pd.DataFrame(
    [{"Deprel": d, "Count": c} for d, c in deprel_counter.items()]
).sort_values("Count", ascending=False)

print("\n=== Deprel of top head (after climbing conj chain) ===")
print(df_deprel_stats)

# ------------------------------
# Save CSV
# ------------------------------
output_folder = "conj_participle_analysis"
os.makedirs(output_folder, exist_ok=True)

df.to_csv(os.path.join(output_folder, "conj_participle_tokens.csv"), index=False)
df_deprel_stats.to_csv(os.path.join(output_folder, "conj_participle_head_deprel_stats.csv"), index=False)

print(f"\n✔ CSV files saved in '{output_folder}'")


=== Deprel of top head (after climbing conj chain) ===
  Deprel  Count
2  advcl     55
0   root     15
1    obl      3
3  xcomp      2
5    obj      2
4  nsubj      1

✔ CSV files saved in 'conj_participle_analysis'


In [5]:
import pandas as pd
from collections import defaultdict
import os

# ------------------------------
# Helper: climb conj chain and count links
# ------------------------------
def get_top_head_and_conj_count(token, token_by_id):
    current = token
    conj_count = 0
    while True:
        head_id = str(current.head)
        if head_id not in token_by_id:
            return None, conj_count
        head = token_by_id[head_id]
        if head.deprel != "conj":
            return head, conj_count
        conj_count += 1
        current = head

# ------------------------------
# Helper: get AUX children
# ------------------------------
def get_aux_children(token, tokens):
    aux_list = []
    for child in tokens:
        if child.head == token.id and child.upos == "AUX":
            aux_list.append(child.lemma)
    return aux_list

# ------------------------------
# Storage
# ------------------------------
records = []

# ------------------------------
# Process corpus
# ------------------------------
for sentence in syntactically_annotated_corpus:
    file_id = sentence.metadata.get("source_filename", "unknown")
    tokens = sentence.get_tokens()
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:
        # only conj participles
        if token.deprel != "conj":
            continue
        if token.upos != "VERB":
            continue
        if not token.feats or token.feats.get("VerbForm") != "Part":
            continue

        # head
        head_id = str(token.head)
        if head_id not in token_by_id:
            continue
        head = token_by_id[head_id]

        if head.upos != "VERB":
            continue
        if not head.feats or head.feats.get("VerbForm") != "Part":
            continue

        # climb to top non-conj head and count intermediate conjs
        top_head, conj_count = get_top_head_and_conj_count(head, token_by_id)
        if not top_head:
            continue

        # collect AUX dependents
        aux_conj = get_aux_children(token, tokens)
        aux_top = get_aux_children(top_head, tokens)

        records.append({
            "File": file_id,
            "ConjLemma": token.lemma,
            "Aux_Conj": "|".join(aux_conj),
            "TopHeadLemma": top_head.lemma,
            "Aux_TopHead": "|".join(aux_top),
            "TopHeadDeprel": top_head.deprel,
            "conj_to_the_conj": conj_count
        })

# ------------------------------
# Build DataFrame
# ------------------------------
df = pd.DataFrame(records)

# Optional: sort by file + conj count
df = df.sort_values(by=["File", "conj_to_the_conj"], ascending=[True, False])

print("\nSample results:")
print(df.head(20))

# ------------------------------
# Save CSV
# ------------------------------
output_folder = "conj_participle_aux_analysis"
os.makedirs(output_folder, exist_ok=True)
df.to_csv(os.path.join(output_folder, "conj_participle_aux_summary.csv"), index=False)

print(f"\n✔ CSV saved in '{output_folder}'")


Sample results:
                   File   ConjLemma Aux_Conj  TopHeadLemma Aux_TopHead  \
0    DD-K35_mptf.conllu      guftan  ēstādan      pursīdan               
1     Dk5-B_mptf.conllu  padīriftan  ēstādan           dēn               
2     Dk5-B_mptf.conllu    gumēxtan           nigerīhistan               
3   GBd-TD1_mptf.conllu   paywastan                    čim               
4   GBd-TD1_mptf.conllu   paywastan                    čim               
5   GBd-TD1_mptf.conllu      madan1                 madan1          h-   
6   GBd-TD1_mptf.conllu      madan1                 madan1          h-   
7   GBd-TD1_mptf.conllu      madan1                 madan1          h-   
8   GBd-TD1_mptf.conllu      madan1                 madan1          h-   
9   GBd-TD1_mptf.conllu      madan1                 madan1          h-   
10  GBd-TD1_mptf.conllu      madan1                 madan1          h-   
11  GBd-TD1_mptf.conllu      madan1                 madan1          h-   
12  GBd-TD1_mptf.conl

In [6]:
import pandas as pd
from collections import defaultdict

# ------------------------------
# Storage for results
# ------------------------------
records = []

# ------------------------------
# Process corpus
# ------------------------------
for sentence in syntactically_annotated_corpus:  # you can switch to my_corpus if needed
    tokens = sentence.get_tokens()
    file_id = sentence.metadata.get("source_filename", "unknown")
    
    token_by_id = {tok.id: tok for tok in tokens if tok.id is not None}
    
    for token in tokens:
        # Only conj tokens that are participles
        if token.deprel != "conj":
            continue
        if token.upos != "VERB" or (not token.feats or token.feats.get("VerbForm") != "Part"):
            continue
        
        # Step 1: climb to top head (first non-conj head)
        current = token
        while True:
            head_tok = token_by_id.get(current.head)
            if not head_tok:
                break
            if head_tok.deprel != "conj":
                top_head = head_tok
                break
            current = head_tok
        else:
            continue  # if no top head, skip
        
        # Step 2: collect aux forms for conj and top head
        def get_aux_forms(tok):
            if not tok:
                return []
            return [child.form for child in tokens if child.head == tok.id and child.upos == "AUX"]
        
        aux_conj = ", ".join(get_aux_forms(token)) or ""
        aux_tophead = ", ".join(get_aux_forms(top_head)) or ""
        
        # Step 3: store record
        records.append({
            "ConjLemma": token.lemma,
            "Aux_Conj": aux_conj,
            "TopHeadLemma": top_head.lemma,
            "Aux_TopHead": aux_tophead,
            "TopHeadDeprel": top_head.deprel,
            "File": file_id
        })

# ------------------------------
# Build DataFrame
# ------------------------------
df_conj_aux = pd.DataFrame(records)

# ------------------------------
# Save CSV
# ------------------------------
output_folder = "conj_part_aux_stats"
import os
os.makedirs(output_folder, exist_ok=True)
df_conj_aux.to_csv(os.path.join(output_folder, "conj_part_aux_forms.csv"), index=False)

print(f"✔ CSV saved in '{output_folder}'")
print(df_conj_aux)

✔ CSV saved in 'conj_part_aux_stats'
     ConjLemma Aux_Conj TopHeadLemma Aux_TopHead TopHeadDeprel  \
0       guftan    istēd        dādan       istēd         ccomp   
1     āstardan    ēstēd      ēstādan                 acl:relcl   
2    payrāstan                kirdan       ēstēd          root   
3       kirdan    ēstēd      wārīdan                     advcl   
4       kirdan     hēnd    winnārdan         būd          root   
..         ...      ...          ...         ...           ...   
132      šudan     hēnd          tēx                     advcl   
133     rustan    ēstād        dīdan                      root   
134     srūdan    ēstēd       kirdan       ēstēd         csubj   
135     rustan    ēstēd        dīdan                      root   
136     srūdan    ēstēd       kirdan       ēstēd    dislocated   

                     File  
0      DD-K35_mptf.conllu  
1      DD-K35_mptf.conllu  
2      DD-K35_mptf.conllu  
3      DD-K35_mptf.conllu  
4     DMX-L19_mptf.conllu  
..

In [ ]:
import pandas as pd
from collections import defaultdict
import os

# ------------------------------
# Storage for results
# ------------------------------
records = []

# ------------------------------
# Process corpus
# ------------------------------
for sentence in syntactically_annotated_corpus:
    tokens = sentence.get_tokens()
    file_id = sentence.metadata.get("source_filename", "unknown")
    
    token_by_id = {tok.id: tok for tok in tokens if tok.id is not None}
    
    for token in tokens:
        # Only conj tokens that are participles
        if token.deprel != "conj":
            continue
        if token.upos != "VERB" or (not token.feats or token.feats.get("VerbForm") != "Part"):
            continue
        
        # Step 1: climb to top head (first non-conj head)
        current = token
        while True:
            head_tok = token_by_id.get(current.head)
            if not head_tok:
                break
            if head_tok.deprel != "conj":
                top_head = head_tok
                break
            current = head_tok
        else:
            continue
        
        # Step 2: collect aux forms
        def get_aux_forms(tok):
            if not tok:
                return []
            return [child.form for child in tokens if child.head == tok.id and child.upos == "AUX"]
        
        aux_conj = ", ".join(get_aux_forms(token)) or ""
        aux_tophead = ", ".join(get_aux_forms(top_head)) or ""
        
        # Step 3: determine order (before / after head)
        if token.id < top_head.id:
            order = "BeforeHead"
        else:
            order = "AfterHead"
        
        # Step 4: store record
        records.append({
            # Conj BEFORE head
            "ConjLemma_Before": token.lemma if order == "BeforeHead" else "",
            "Aux_Conj_Before": aux_conj if order == "BeforeHead" else "",
            
            # Head
            "TopHeadLemma": top_head.lemma,
            "Aux_TopHead": aux_tophead,
            "TopHeadDeprel": top_head.deprel,
            
            # Conj AFTER head
            "ConjLemma_After": token.lemma if order == "AfterHead" else "",
            "Aux_Conj_After": aux_conj if order == "AfterHead" else "",
            
            "Order": order,
            "File": file_id
        })

# ------------------------------
# Build DataFrame
# ------------------------------
df_conj_aux = pd.DataFrame(records)

# ------------------------------
# Save CSV
# ------------------------------
output_folder = "conj_part_aux_stats"
os.makedirs(output_folder, exist_ok=True)

df_conj_aux.to_csv(
    os.path.join(output_folder, "conj_part_aux_forms-order.csv"),
    index=False
)

print(f"✔ CSV saved in '{output_folder}'")
print(df_conj_aux)

✔ CSV saved in 'conj_part_aux_stats'
    ConjLemma_Before Aux_Conj_Before TopHeadLemma Aux_TopHead TopHeadDeprel  \
0                                           dādan       istēd         ccomp   
1                                         ēstādan                 acl:relcl   
2                                          kirdan       ēstēd          root   
3                                         wārīdan                     advcl   
4                                       winnārdan         būd          root   
..               ...             ...          ...         ...           ...   
132                                           tēx                     advcl   
133                                         dīdan                      root   
134                                        kirdan       ēstēd         csubj   
135                                         dīdan                      root   
136                                        kirdan       ēstēd    dislocated   

    ConjLemma_

In [7]:
# ------------------------------
# 1️⃣ Same auxiliary forms
# ------------------------------
same_aux = df_conj_aux[
    (df_conj_aux["Aux_Conj"] != "") &
    (df_conj_aux["Aux_TopHead"] != "") &
    (df_conj_aux["Aux_Conj"] == df_conj_aux["Aux_TopHead"])
]
print("\n=== Conj and TopHead with the same AUX ===")
print(same_aux.head(20))


=== Conj and TopHead with the same AUX ===
     ConjLemma Aux_Conj    TopHeadLemma Aux_TopHead TopHeadDeprel  \
0       guftan    istēd           dādan       istēd         ccomp   
85   handāxtan      hād        dānistan         hād         advcl   
96      hištan       hē           dādan          hē          root   
110   xwārīdan    ēstēd          kirdan       ēstēd     acl:relcl   
111      dādan    ēstēd          yaštan       ēstēd     acl:relcl   
112     kirdan    ēstēd          yaštan       ēstēd     acl:relcl   
113     madan1      hom  nēryōsangyazad         hom         advcl   
124    tāšīdan      ham      brēhēnīdan         ham         ccomp   
125    tāšīdan       hē      brēhēnīdan          hē          root   
131     baxtan     hēnd         xwāstan        hēnd         csubj   
134     srūdan    ēstēd          kirdan       ēstēd         csubj   
136     srūdan    ēstēd          kirdan       ēstēd    dislocated   

                     File  
0      DD-K35_mptf.conllu  
85

In [8]:
# ------------------------------
# 2️⃣ Both have AUX, but different forms
# ------------------------------
both_diff_aux = df_conj_aux[
    (df_conj_aux["Aux_Conj"] != "") &
    (df_conj_aux["Aux_TopHead"] != "") &
    (df_conj_aux["Aux_Conj"] != df_conj_aux["Aux_TopHead"])
]
print("\n=== Conj and TopHead both have AUX but different forms ===")
print(both_diff_aux.head(20))


=== Conj and TopHead both have AUX but different forms ===
    ConjLemma Aux_Conj TopHeadLemma Aux_TopHead TopHeadDeprel  \
4      kirdan     hēnd    winnārdan         būd          root   
26     madan1      ast       madan1        hēnd     parataxis   
106   nihādan    ēstēd       guftan      abāyēd     acl:relcl   
130     būdan     hēnd         ādūg       bawēd          root   

                    File  
4    DMX-L19_mptf.conllu  
26   GBd-TD1_mptf.conllu  
106  GBd-TD1_mptf.conllu  
130        WZ_K35.conllu  


In [9]:
# ------------------------------
# 3️⃣ Conj has AUX, TopHead does NOT
# ------------------------------
conj_has_aux = df_conj_aux[
    (df_conj_aux["Aux_Conj"] != "") &
    (df_conj_aux["Aux_TopHead"] == "")
]
print("\n=== Conj has AUX, TopHead does NOT ===")
print(conj_has_aux.head(20))


=== Conj has AUX, TopHead does NOT ===
                ConjLemma Aux_Conj TopHeadLemma Aux_TopHead TopHeadDeprel  \
1                āstardan    ēstēd      ēstādan                 acl:relcl   
3                  kirdan    ēstēd      wārīdan                     advcl   
7              padīriftan    ēstēd      wizīdan                 acl:relcl   
12              paywastan     hēnd      tuhīgīh                     advcl   
13                raftan1     hēnd      ēstādan                      root   
92              winnārdan     hēnd      ēstādan                      root   
93                 bōxtan     hēnd      ōbastan                     ccomp   
102                bištan    bawēd    rāmēnīdan                      root   
103                bištan    bawēd    rāmēnīdan                      root   
104                bištan    bawēd    rāmēnīdan                      root   
105                bištan    bawēd    rāmēnīdan                      root   
107              nišastan    ēstēd  

In [10]:
# ------------------------------
# 4️⃣ TopHead has AUX, Conj does NOT
# ------------------------------
tophead_has_aux = df_conj_aux[
    (df_conj_aux["Aux_Conj"] == "") &
    (df_conj_aux["Aux_TopHead"] != "")
]
print("\n=== TopHead has AUX, Conj does NOT ===")
print(tophead_has_aux.head(20))


=== TopHead has AUX, Conj does NOT ===
    ConjLemma Aux_Conj TopHeadLemma Aux_TopHead TopHeadDeprel  \
2   payrāstan                kirdan       ēstēd          root   
6   payrāstan               tāšīdan       ēstād          root   
8    gumēxtan              gumēxtan       ēstēd         csubj   
16     madan1                madan1        hēnd     parataxis   
17     madan1                madan1        hēnd     parataxis   
18     madan1                madan1        hēnd     parataxis   
19     madan1                madan1        hēnd     parataxis   
20     madan1                madan1        hēnd     parataxis   
21     madan1                madan1        hēnd     parataxis   
22     madan1                madan1        hēnd     parataxis   
23     madan1                madan1        hēnd     parataxis   
24     madan1                madan1        hēnd     parataxis   
25     madan1                madan1        hēnd     parataxis   
27     madan1                madan1        hēnd   

In [11]:
import pandas as pd

# Temporarily show all rows and columns
with pd.option_context('display.max_rows', None, 'display.max_columns', None):

    # ------------------------------
    # 1️⃣ Conj and TopHead have the same AUX
    # ------------------------------
    same_aux = df_conj_aux[
        (df_conj_aux["Aux_Conj"] != "") &
        (df_conj_aux["Aux_TopHead"] != "") &
        (df_conj_aux["Aux_Conj"] == df_conj_aux["Aux_TopHead"])
    ]
    print("\n=== 1️⃣ Conj and TopHead have the SAME AUX ===")
    print(same_aux)

    # ------------------------------
    # 2️⃣ Conj and TopHead both have AUX but different
    # ------------------------------
    diff_aux = df_conj_aux[
        (df_conj_aux["Aux_Conj"] != "") &
        (df_conj_aux["Aux_TopHead"] != "") &
        (df_conj_aux["Aux_Conj"] != df_conj_aux["Aux_TopHead"])
    ]
    print("\n=== 2️⃣ Conj and TopHead have DIFFERENT AUX ===")
    print(diff_aux)

    # ------------------------------
    # 3️⃣ Conj has AUX, TopHead does NOT
    # ------------------------------
    conj_has_aux_only = df_conj_aux[
        (df_conj_aux["Aux_Conj"] != "") &
        (df_conj_aux["Aux_TopHead"] == "")
    ]
    print("\n=== 3️⃣ Conj has AUX, TopHead does NOT ===")
    print(conj_has_aux_only)

    # ------------------------------
    # 4️⃣ TopHead has AUX, Conj does NOT
    # ------------------------------
    tophead_has_aux_only = df_conj_aux[
        (df_conj_aux["Aux_Conj"] == "") &
        (df_conj_aux["Aux_TopHead"] != "")
    ]
    print("\n=== 4️⃣ TopHead has AUX, Conj does NOT ===")
    print(tophead_has_aux_only)


=== 1️⃣ Conj and TopHead have the SAME AUX ===
     ConjLemma Aux_Conj    TopHeadLemma Aux_TopHead TopHeadDeprel  \
0       guftan    istēd           dādan       istēd         ccomp   
85   handāxtan      hād        dānistan         hād         advcl   
96      hištan       hē           dādan          hē          root   
110   xwārīdan    ēstēd          kirdan       ēstēd     acl:relcl   
111      dādan    ēstēd          yaštan       ēstēd     acl:relcl   
112     kirdan    ēstēd          yaštan       ēstēd     acl:relcl   
113     madan1      hom  nēryōsangyazad         hom         advcl   
124    tāšīdan      ham      brēhēnīdan         ham         ccomp   
125    tāšīdan       hē      brēhēnīdan          hē          root   
131     baxtan     hēnd         xwāstan        hēnd         csubj   
134     srūdan    ēstēd          kirdan       ēstēd         csubj   
136     srūdan    ēstēd          kirdan       ēstēd    dislocated   

                     File  
0      DD-K35_mptf.conllu 

In [12]:
import pandas as pd

# Columns we care about
cols = ["ConjLemma", "Aux_Conj", "TopHeadLemma", "Aux_TopHead", "TopHeadDeprel", "File"]

# Group by lemma pair
grouped = df_conj_aux.groupby(["ConjLemma", "TopHeadLemma"])

agg_records = []

for (conj, head), group in grouped:
    # Get unique AUX combinations as tuples (Aux_Conj, Aux_TopHead)
    aux_variations = set((row["Aux_Conj"], row["Aux_TopHead"]) for _, row in group.iterrows())
    
    # Only keep lemma pairs with more than one AUX combination
    if len(aux_variations) > 1:
        # Add all rows for this lemma pair that represent each unique AUX combination
        seen = set()
        for _, row in group.iterrows():
            key = (row["Aux_Conj"], row["Aux_TopHead"])
            if key not in seen:
                agg_records.append(row[cols].to_dict())
                seen.add(key)

# Build final DataFrame
df_true_variations = pd.DataFrame(agg_records)

# Sort nicely
df_true_variations = df_true_variations.sort_values(by=["ConjLemma", "TopHeadLemma", "Aux_Conj", "Aux_TopHead"]).reset_index(drop=True)

# Show all
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(df_true_variations)

# Save CSV
df_true_variations.to_csv("conj_head_aux_true_variations_strict.csv", index=False)
print("\n✔ Saved only STRICT TRUE variations by lemma pair to 'conj_head_aux_true_variations_strict.csv'")

   ConjLemma Aux_Conj TopHeadLemma Aux_TopHead TopHeadDeprel  \
0     bastan                bastan       ēstēd          root   
1     bastan                bastan      ēstēnd         advcl   
2     bištan             rāmēnīdan       bawēd          root   
3     bištan    bawēd    rāmēnīdan                      root   
4     kirdan                yaštan                      root   
5     kirdan    ēstēd       yaštan       ēstēd     acl:relcl   
6     madan1                madan1        hēnd     parataxis   
7     madan1                madan1       ēstēd     acl:relcl   
8     madan1      ast       madan1        hēnd     parataxis   
9     rustan    ēstād        dīdan                      root   
10    rustan    ēstēd        dīdan                      root   
11   tāšīdan      ham   brēhēnīdan         ham         ccomp   
12   tāšīdan       hē   brēhēnīdan          hē          root   

                    File  
0    GBd-TD1_mptf.conllu  
1    GBd-TD1_mptf.conllu  
2    GBd-TD1_mptf.conl